# 📊 Prediksi Harga Produk Tokopedia
## Perbandingan Algoritma: Linear Regression, Decision Tree, Random Forest

**Tujuan:** Memprediksi harga jual produk berdasarkan fitur:
- Rating produk
- Diskon (%)
- Jumlah ulasan
- Jumlah terjual

**Alur Kerja:**
1. Load & Eksplorasi Data (EDA)
2. Preprocessing
3. Analisis Korelasi Fitur
4. Split Data 80:20
5. Training 3 Model Regresi
6. Evaluasi & Perbandingan Model
7. Visualisasi Hasil Prediksi
8. Kesimpulan

## 📦 1. Setup & Load Data

In [ ]:
!pip install -q matplotlib seaborn pandas numpy scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('✅ Library siap!')

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload file CSV Tokopedia

import io
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'✅ Dataset: {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom')
print('\nKolom tersedia:', df_raw.columns.tolist())
df_raw.head(5)

## 🔍 2. Eksplorasi Data Awal (EDA)

In [ ]:
print('📋 INFO DATASET:')
print(df_raw.info())
print('\n📊 STATISTIK DESKRIPTIF:')
df_raw.describe()

In [ ]:
print('❓ MISSING VALUES per kolom:')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

## 🧹 3. Preprocessing

In [ ]:
df = df_raw.copy()

# ── Parse Terjual → angka
def parse_terjual(x):
    if pd.isna(x): return np.nan
    x = str(x).lower().replace('+','').replace(' terjual','').strip()
    mult = 1
    if 'rb' in x:  x = x.replace('rb','').strip(); mult = 1_000
    elif 'jt' in x: x = x.replace('jt','').strip(); mult = 1_000_000
    try: return float(x) * mult
    except: return np.nan

# ── Parse Ulasan → angka
def parse_ulasan(x):
    if pd.isna(x): return 0
    x = str(x).lower().replace(' ulasan','').replace('+','').replace(',','').strip()
    try: return float(x)
    except: return 0

# Konversi kolom
df['Terjual_num'] = df['Terjual'].apply(parse_terjual)
df['Ulasan_num']  = df['Jumlah Ulasan'].apply(parse_ulasan)
df['Harga']       = pd.to_numeric(df['Harga (IDR)'], errors='coerce')
df['Diskon']      = pd.to_numeric(df['Diskon (%)'], errors='coerce').fillna(0)
df['Rating']      = pd.to_numeric(df['Rating'], errors='coerce')

# Hapus baris dengan nilai kosong pada kolom penting
df = df.dropna(subset=['Harga', 'Rating', 'Terjual_num'])

# Hapus harga = 0 dan outlier ekstrem (1% teratas)
df = df[df['Harga'] > 0]
Q99 = df['Harga'].quantile(0.99)
df = df[df['Harga'] <= Q99]

# Hapus rating = 0
df = df[df['Rating'] > 0]

print(f'✅ Data setelah preprocessing: {len(df):,} produk')
print(f'   Harga min : Rp {int(df["Harga"].min()):,}')
print(f'   Harga max : Rp {int(df["Harga"].max()):,}')
print(f'   Harga mean: Rp {int(df["Harga"].mean()):,}')
df[['Harga','Diskon','Rating','Terjual_num','Ulasan_num']].describe().round(2)

In [ ]:
# Visualisasi distribusi variabel
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribusi Variabel Setelah Preprocessing', fontsize=13, fontweight='bold')

kolom_plot = [
    ('Harga',       'Harga (IDR)',          'steelblue'),
    ('Diskon',      'Diskon (%)',            'coral'),
    ('Rating',      'Rating',               'mediumseagreen'),
    ('Terjual_num', 'Jumlah Terjual',        'mediumpurple'),
    ('Ulasan_num',  'Jumlah Ulasan',         'darkorange'),
]

for idx, (col, label, color) in enumerate(kolom_plot):
    ax = axes[idx // 3][idx % 3]
    data = df[col][df[col] < df[col].quantile(0.97)]
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribusi {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Frekuensi')

axes[1][2].axis('off')
plt.tight_layout()
plt.show()

## 🔗 4. Analisis Korelasi Fitur terhadap Harga

In [ ]:
# Heatmap korelasi
kolom_korelasi = ['Harga','Diskon','Rating','Terjual_num','Ulasan_num']
corr = df[kolom_korelasi].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis Korelasi Fitur', fontsize=13, fontweight='bold')

# Heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, ax=axes[0], vmin=-1, vmax=1,
            linewidths=0.5, square=True)
axes[0].set_title('Heatmap Korelasi')

# Bar korelasi terhadap Harga
korelasi_harga = corr['Harga'].drop('Harga').sort_values()
colors_bar = ['#E74C3C' if v < 0 else '#2ECC71' for v in korelasi_harga]
bars = axes[1].barh(korelasi_harga.index, korelasi_harga.values, color=colors_bar)
axes[1].bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('Korelasi dengan Harga')
axes[1].set_title('Kekuatan Korelasi Fitur → Harga\n(hijau=positif, merah=negatif)')
axes[1].set_xlim(-0.3, 0.3)

plt.tight_layout()
plt.show()

print('\n📌 Interpretasi:')
for fitur, r in korelasi_harga.items():
    kuat = 'Kuat' if abs(r)>=0.5 else ('Sedang' if abs(r)>=0.3 else 'Lemah')
    arah = 'positif' if r > 0 else 'negatif'
    print(f'   {fitur:15s} → r={r:+.3f} ({kuat}, {arah})')

## ✂️ 5. Persiapan Fitur & Split Data 80:20

In [ ]:
# Fitur (X) dan target (y)
FITUR = ['Rating', 'Diskon', 'Terjual_num', 'Ulasan_num']
TARGET = 'Harga'

X = df[FITUR].copy()
y = df[TARGET].copy()

# Split 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standarisasi (untuk Linear Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Split Data 80:20')
print(f'   Total data  : {len(X):,} produk')
print(f'   Training    : {len(X_train):,} produk ({len(X_train)/len(X)*100:.0f}%)')
print(f'   Testing     : {len(X_test):,} produk ({len(X_test)/len(X)*100:.0f}%)')
print(f'\n   Fitur (X)   : {FITUR}')
print(f'   Target (y)  : {TARGET}')

## 🤖 6. Training 3 Model Regresi

In [ ]:
# ── Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('✅ Model 1: Linear Regression — selesai ditraining')
print('   Koefisien fitur:')
for f, c in zip(FITUR, lr.coef_):
    print(f'     {f:15s}: {c:+.2f}')
print(f'   Intercept: {lr.intercept_:.2f}')

In [ ]:
# ── Model 2: Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('✅ Model 2: Decision Tree Regressor — selesai ditraining')
print(f'   Max depth  : {dt.max_depth}')
print(f'   Jumlah leaf: {dt.get_n_leaves()}')

In [ ]:
# ── Model 3: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('✅ Model 3: Random Forest Regressor — selesai ditraining')
print(f'   Jumlah tree: {rf.n_estimators}')
print(f'   Max depth  : {rf.max_depth}')

## 📏 7. Evaluasi & Perbandingan Model

In [ ]:
# Fungsi evaluasi
def evaluasi(nama, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Model': nama, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

hasil = pd.DataFrame([
    evaluasi('Linear Regression', y_test, y_pred_lr),
    evaluasi('Decision Tree',     y_test, y_pred_dt),
    evaluasi('Random Forest',     y_test, y_pred_rf),
])
hasil = hasil.set_index('Model')

print('='*55)
print('  📊 HASIL EVALUASI 3 MODEL')
print('='*55)
print(hasil.round(2).to_string())
print('='*55)
print('\nKeterangan:')
print('  MAE  = Mean Absolute Error  (makin kecil makin baik)')
print('  RMSE = Root Mean Squared Error (makin kecil makin baik)')
print('  R²   = Koefisien Determinasi  (makin mendekati 1 makin baik)')

best_model = hasil['R²'].idxmax()
print(f'\n🏆 Model terbaik: {best_model} (R² = {hasil.loc[best_model,"R²"]:.4f})')

In [ ]:
# Visualisasi perbandingan metrik
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Perbandingan Performa 3 Model Regresi', fontsize=13, fontweight='bold')

model_names = hasil.index.tolist()
colors_m = ['#3498DB', '#E67E22', '#2ECC71']

# MAE
bars0 = axes[0].bar(model_names, hasil['MAE'], color=colors_m)
axes[0].bar_label(bars0, fmt='%.0f', padding=3, fontsize=9)
axes[0].set_title('MAE (lebih kecil = lebih baik)')
axes[0].set_ylabel('MAE (IDR)')
axes[0].tick_params(axis='x', rotation=15)

# RMSE
bars1 = axes[1].bar(model_names, hasil['RMSE'], color=colors_m)
axes[1].bar_label(bars1, fmt='%.0f', padding=3, fontsize=9)
axes[1].set_title('RMSE (lebih kecil = lebih baik)')
axes[1].set_ylabel('RMSE (IDR)')
axes[1].tick_params(axis='x', rotation=15)

# R²
bars2 = axes[2].bar(model_names, hasil['R²'], color=colors_m)
axes[2].bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)
axes[2].set_title('R² (lebih tinggi = lebih baik)')
axes[2].set_ylabel('R² Score')
axes[2].set_ylim(0, 1.1)
axes[2].tick_params(axis='x', rotation=15)

# Highlight bar terbaik
best_idx = model_names.index(best_model)
for bars in [bars0, bars1, bars2]:
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.show()

## 📈 8. Visualisasi Hasil Prediksi

In [ ]:
# Actual vs Predicted untuk 3 model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Actual vs Predicted Harga — 3 Model', fontsize=13, fontweight='bold')

models_pred = [
    ('Linear Regression', y_pred_lr, '#3498DB'),
    ('Decision Tree',     y_pred_dt, '#E67E22'),
    ('Random Forest',     y_pred_rf, '#2ECC71'),
]

# Ambil sample 500 titik supaya tidak terlalu padat
np.random.seed(42)
sample_idx = np.random.choice(len(y_test), min(500, len(y_test)), replace=False)
y_test_arr = np.array(y_test)

for ax, (nama, y_pred, color) in zip(axes, models_pred):
    y_pred_arr = np.array(y_pred)
    ax.scatter(y_test_arr[sample_idx], y_pred_arr[sample_idx],
               alpha=0.4, s=15, color=color)
    # Garis ideal (perfect prediction)
    lims = [y_test_arr.min(), y_test_arr.max()]
    ax.plot(lims, lims, 'r--', lw=2, label='Prediksi Sempurna')
    r2 = r2_score(y_test, y_pred)
    ax.set_xlabel('Harga Aktual (IDR)')
    ax.set_ylabel('Harga Prediksi (IDR)')
    ax.set_title(f'{nama}\nR² = {r2:.4f}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}rb'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}rb'))
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance & Residual Analysis', fontsize=13, fontweight='bold')

# Feature importance RF
importances = pd.Series(rf.feature_importances_, index=FITUR).sort_values(ascending=True)
bars = axes[0].barh(importances.index, importances.values,
                    color=sns.color_palette('viridis', len(FITUR)))
axes[0].bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
axes[0].set_xlabel('Importance Score')
axes[0].set_title('Feature Importance\n(Random Forest — fitur mana paling berpengaruh?)')

# Residual plot (RF)
residual = np.array(y_test) - np.array(y_pred_rf)
axes[1].scatter(y_pred_rf[:500], residual[:500], alpha=0.3, s=15, color='#9B59B6')
axes[1].axhline(0, color='red', lw=2, linestyle='--')
axes[1].set_xlabel('Prediksi Harga (IDR)')
axes[1].set_ylabel('Residual (Aktual - Prediksi)')
axes[1].set_title('Residual Plot — Random Forest\n(titik dekat garis merah = prediksi akurat)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x/1000)}rb'))

plt.tight_layout()
plt.show()

## 🎯 9. Contoh Prediksi Harga Produk Baru

In [ ]:
# Simulasi: prediksi harga untuk produk baru
# Ganti nilai di bawah sesuai produk yang ingin diprediksi

produk_baru = pd.DataFrame({
    'Rating':      [4.9, 4.5, 5.0],
    'Diskon':      [10,  0,   25 ],
    'Terjual_num': [500, 50,  2000],
    'Ulasan_num':  [30,  5,   150 ],
})

label_produk = ['Produk A (Rating tinggi, diskon 10%)',
                'Produk B (Rating sedang, tanpa diskon)',
                'Produk C (Rating sempurna, diskon besar, banyak terjual)']

# Prediksi pakai model terbaik (RF)
prediksi_rf = rf.predict(produk_baru)
prediksi_lr = lr.predict(scaler.transform(produk_baru))
prediksi_dt = dt.predict(produk_baru)

print('🎯 PREDIKSI HARGA PRODUK BARU:')
print('='*65)
for i, label in enumerate(label_produk):
    print(f'\n{label}')
    print(f'  Linear Regression : Rp {int(prediksi_lr[i]):,}')
    print(f'  Decision Tree     : Rp {int(prediksi_dt[i]):,}')
    print(f'  Random Forest     : Rp {int(prediksi_rf[i]):,}  ← rekomendasi (model terbaik)')
print('='*65)

## 📋 10. Kesimpulan

In [ ]:
print('='*65)
print('  📝 KESIMPULAN PROYEK')
print('='*65)
print(f'''
1. DATASET
   • Sumber data  : Tokopedia (web scraping)
   • Jumlah data  : {len(df):,} produk (setelah preprocessing)
   • Target       : Prediksi Harga Jual (IDR)
   • Fitur        : Rating, Diskon, Terjual, Ulasan

2. PREPROCESSING
   • Parse kolom teks (Terjual, Ulasan) → numerik
   • Hapus missing value & outlier (>99 persentil)
   • Standarisasi fitur untuk Linear Regression
   • Split 80% training / 20% testing

3. HASIL EVALUASI MODEL
''')
print(hasil.round(4).to_string())
print(f'''
4. KESIMPULAN
   🏆 Model terbaik : {best_model}
      → R² = {hasil.loc[best_model, "R²"]:.4f} (menjelaskan {hasil.loc[best_model, "R²"]*100:.1f}% variasi harga)
      → MAE = Rp {int(hasil.loc[best_model, "MAE"]):,} (rata-rata selisih prediksi vs aktual)

   📌 Fitur paling berpengaruh terhadap harga:
      {importances.idxmax()} (importance = {importances.max():.3f})

   ⚠️  Catatan:
      Model ini memprediksi harga berdasarkan performa produk.
      Semakin tinggi penjualan dan ulasan, semakin akurat prediksinya.
''')
print('='*65)

## 📤 11. Export CSV untuk Tableau

Jalankan cell ini untuk menghasilkan file CSV yang siap dibuka di **Tableau Public**.

In [ ]:
from google.colab import files

# ── CSV 1: Aktual vs Prediksi Linear Regression
df_pred = X_test.copy()
df_pred['Harga_Aktual']   = y_test.values.astype(int)
df_pred['Harga_Prediksi'] = y_pred_lr.astype(int)
df_pred['Selisih']        = df_pred['Harga_Aktual'] - df_pred['Harga_Prediksi']
df_pred['Selisih_Abs']    = df_pred['Selisih'].abs()
df_pred['Status']         = df_pred['Selisih'].apply(
    lambda x: 'Prediksi Terlalu Tinggi' if x < -10000
    else ('Prediksi Terlalu Rendah' if x > 10000 else 'Akurat')
)
# Bulatkan semua angka desimal agar Tableau bisa baca
df_pred['Rating']      = df_pred['Rating'].round(1)
df_pred['Diskon']      = df_pred['Diskon'].round(0).astype(int)
df_pred['Terjual_num'] = df_pred['Terjual_num'].round(0).astype(int)
df_pred['Ulasan_num']  = df_pred['Ulasan_num'].round(0).astype(int)
df_pred['Selisih_Abs'] = df_pred['Selisih_Abs'].round(0).astype(int)
df_pred.to_csv('tableau_prediksi_harga.csv', index=False)
print('✅ File 1 siap: tableau_prediksi_harga.csv')
print(f'   {len(df_pred):,} baris | kolom: {df_pred.columns.tolist()}')

# ── CSV 2: Perbandingan Performa 3 Model
df_model = hasil.reset_index()
df_model.columns = ['Model', 'MAE', 'RMSE', 'R2']
df_model.to_csv('tableau_performa_model.csv', index=False)
print('\n✅ File 2 siap: tableau_performa_model.csv')
print(f'   {len(df_model)} baris | kolom: {df_model.columns.tolist()}')

# ── CSV 3: Penjualan per Kota
city_tableau = df.dropna(subset=['Terjual_num']).groupby('Lokasi Toko').agg(
    Jumlah_Produk  = ('Harga',        'count'),
    Total_Terjual  = ('Terjual_num',  'sum'),
    Revenue_Est    = ('Est_Revenue',  'sum') if 'Est_Revenue' in df.columns else ('Harga', 'sum'),
    Harga_Median   = ('Harga',        'median'),
    Rating_Avg     = ('Rating',       'mean'),
).reset_index()
city_tableau.to_csv('tableau_penjualan_kota.csv', index=False)
print('\n✅ File 3 siap: tableau_penjualan_kota.csv')
print(f'   {len(city_tableau):,} baris | kolom: {city_tableau.columns.tolist()}')

# ── Download semua file
print('\n📥 Mendownload semua file CSV...')
files.download('tableau_prediksi_harga.csv')
files.download('tableau_performa_model.csv')
files.download('tableau_penjualan_kota.csv')
print('✅ Semua file berhasil didownload!')

## 📊 Panduan Membuat Visualisasi di Tableau Public

Setelah file CSV didownload, buka **Tableau Public** dan ikuti langkah berikut:

---

### 📁 File 1: `tableau_prediksi_harga.csv`
**Visualisasi yang dibuat:**
- **Scatter Plot** → Harga Aktual (X) vs Harga Prediksi (Y) → drag `Harga_Aktual` ke Columns, `Harga_Prediksi` ke Rows, warnai berdasarkan `Status`
- **Bar Chart** → Rata-rata Selisih per Rating → drag `Rating` ke Columns, `AVG(Selisih_Abs)` ke Rows

---

### 📁 File 2: `tableau_performa_model.csv`
**Visualisasi yang dibuat:**
- **Bar Chart** → Perbandingan MAE, RMSE, R² → drag `Model` ke Columns, lalu buat 3 sheet terpisah untuk MAE / RMSE / R2

---

### 📁 File 3: `tableau_penjualan_kota.csv`
**Visualisasi yang dibuat:**
- **Map** → drag `Lokasi Toko` ke canvas (Tableau auto-detect kota Indonesia) → warnai berdasarkan `Total_Terjual`
- **Bar Chart** → Top 15 kota berdasarkan `Total_Terjual`

---

**💡 Tips:** Gabungkan semua sheet ke dalam 1 **Dashboard** di Tableau untuk presentasi!